In [ ]:
"""
Script to search for LPG sales points in Dar es Salaam (Tanzania) using:
- OpenStreetMap (Overpass API) – main
- Google Places API (optional, with flag)

Output: CSV, JSON, GeoPackage (EPSG:3857)
"""

import requests
import time
import csv
import json
import os
import math
from datetime import datetime
from collections import Counter

# ========================= CONFIGURATION =========================
USE_GOOGLE = True
API_KEY = "yourkey"

CITY_NAME = "dar_es_salaam"          # ← CHANGE HERE for each city (use underscore)
DAR_BBOX = {                         # ← MODIFY these coordinates for the new city
    'min_lat': -7.2,
    'max_lat': -6.4,
    'min_lon': 38.9,
    'max_lon': 39.6
}

# Root folder of datasets, with subfolder for the city
DATA_DIR = os.path.join("other_city", CITY_NAME)
os.makedirs(DATA_DIR, exist_ok=True)

STEP = 0.2        # degrees (~22 km)
RADIUS = 50000    # metres (used only for Google)


# Keywords for Google (only if USE_GOOGLE = True)
# Keywords for resellers
RESELLER_KEYWORDS = {
    'en': [
        "LPG refill station", "gas station with LPG", "LPG filling point",
        "LPG dealer", "gas cylinder exchange", "cooking gas", "LPG", "butane gas", "butane cylinder"
    ],
    'fr': [
        "station de remplissage de GPL", "station-service avec GPL",
        "point de remplissage de GPL", "revendeur de GPL",
        "échange de bouteilles de gaz", "gaz de cuisine", "GPL", "gaz butane", "bouteille de gaz butane"
    ],
    'pt': [
        "estação de recarga de GLP", "posto de gasolina com GLP",
        "ponto de enchimento de GLP", "revendedor de GLP",
        "troca de botijão de gás", "gás de cozinha", "GLP", "gás butano", "botija de gás butano"
    ],
    'sw': [
        "kituo cha kujaza gesi", "kituo cha mafuta chenye GLP",
        "sehemu ya kujaza GLP", "muuzaji wa GLP",
        "kubadilisha mitungi ya gesi", "gesi ya kupikia", "GLP", "gesi ya butane", "mtungi wa gesi ya butane"
    ]
}

# Keyword for filling
FILLING_KEYWORDS = {
    'en': [
        "LPG cylinder filling plant", "LPG bottling plant",
        "gas bottling plant", "LPG filling plant",
        "cylinder filling station", "LPG bottling station"
    ],
    'fr': [
        "usine de remplissage de bouteilles de GPL", "usine d'embouteillage de GPL",
        "usine d'embouteillage de gaz", "usine de remplissage de GPL",
        "station de remplissage de bouteilles", "station d'embouteillage de GPL"
    ],
    'pt': [
        "fábrica de enchimento de botijas de GPL", "fábrica de engarrafamento de GPL",
        "fábrica de engarrafamento de gás", "fábrica de enchimento de GPL",
        "estação de enchimento de botijas", "estação de engarrafamento de GPL"
    ],
    'sw': [
        "kiwanda cha kujaza mitungi ya GPL", "kiwanda cha kutengeneza GPL",
        "kiwanda cha kutengeneza gesi", "kiwanda cha kujaza GPL",
        "kituo cha kujaza mitungi", "kituo cha kutengeneza GPL"
    ]
}

# ==================== OSM FUNCTIONS ====================
def osm_search(lat, lon, radius=50000):
    delta = radius / 111000.0
    min_lat = lat - delta
    max_lat = lat + delta
    min_lon = lon - delta
    max_lon = lon + delta

    query = f"""
    [out:json];
    (
      node["amenity"="fuel"]["fuel:lpg"~"yes|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["amenity"="fuel"]["fuel:lpg"~"yes|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["amenity"="fuel"]["fuel:GPL"~"yes|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["amenity"="fuel"]["fuel:GPL"~"yes|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["shop"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["shop"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["shop"="gas_cooking"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["shop"="gas_cooking"]({min_lat},{min_lon},{max_lat},{max_lon});

    
      node["industrial"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["man_made"="storage_tank"]["content"~"lpg|gpl|glp"]({min_lat},{min_lon},{max_lat},{max_lon});

    );
    out center;
    """

    url = "https://overpass-api.de/api/interpreter"
    headers = {
        'User-Agent': 'LPG-Research-Script/1.0 (contact: your-email@example.com)',
        'Accept': 'application/json'
    }
    try:
        response = requests.post(url, data={'data': query}, headers=headers, timeout=30)
        if response.status_code != 200:
            print(f"  ⚠️ OSM HTTP error {response.status_code}: {response.text[:100]}")
            return []
        data = response.json()
    except Exception as e:
        print(f"  ⚠️ OSM request error: {e}")
        return []

    results = []
    for elem in data.get('elements', []):
        if elem['type'] == 'node':
            lat_place = elem.get('lat')
            lon_place = elem.get('lon')
        else:  # way
            center = elem.get('center', {})
            lat_place = center.get('lat')
            lon_place = center.get('lon')

        if lat_place is None or lon_place is None:
            continue

        tags = elem.get('tags', {})

        # Determine point type (fuel_station, shop, etc.)
        if 'amenity' in tags and tags['amenity'] == 'fuel':
            ptype = 'fuel_station'
        elif 'shop' in tags:
            ptype = tags['shop']
        else:
            ptype = 'lpg_related'

        # Determine category: filling if it matches the new tags
        if (tags.get('industrial') == 'gas' or
            (tags.get('man_made') == 'storage_tank' and
             tags.get('content', '').lower() in ['lpg', 'gpl', 'glp'])):
            category = 'filling'
        else:
            category = 'reseller'

        name = tags.get('name', '')

        results.append({
            'place_id': f"osm_{elem['id']}",
            'name': name,
            'address': '',
            'lat': lat_place,
            'lng': lon_place,
            'types': ptype,
            'keyword': 'osm',
            'search_lat': lat,
            'search_lon': lon,
            'rating': None,
            'user_ratings_total': 0,
            'source': 'osm',
            'language': '',
            'category': category    # <-- now used everywhere
        })
    return results


# ==================== GOOGLE FUNCTIONS (optional) ====================

def google_nearby_search(lat, lon, keyword, lang, category):
    """Google Places Nearby Search call."""
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    results = []
    params = {
        'location': f"{lat},{lon}",
        'radius': RADIUS,
        'keyword': keyword,
        'language': lang,
        'key': API_KEY
    }
    while True:
        resp = requests.get(url, params=params)
        data = resp.json()
        if data['status'] not in ['OK', 'ZERO_RESULTS']:
            error_msg = data.get('error_message', '')
            print(f"  ⚠️ Google status {data['status']} for {keyword}_{lang}: {error_msg}")
            break
        for place in data.get('results', []):
            results.append({
                'place_id': place['place_id'],
                'name': place.get('name', ''),
                'address': place.get('vicinity', ''),
                'lat': place['geometry']['location']['lat'],
                'lng': place['geometry']['location']['lng'],
                'types': ', '.join(place.get('types', [])),
                'keyword': f"{keyword}_{lang}",
                'search_lat': lat,
                'search_lon': lon,
                'rating': place.get('rating'),
                'user_ratings_total': place.get('user_ratings_total', 0),
                'source': 'google',
                'language': lang,
                'category': category     
            })
            
        if 'next_page_token' in data:
            params['pagetoken'] = data['next_page_token']
            time.sleep(2)
        else:
            break
    return results

# ==================== SAVING FUNCTIONS ====================

def save_results(all_results, filling_results, reseller_results, data_dir, prefix):
    """Save CSV, JSON, and two GeoPackage files (filling + reseller)."""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    base = os.path.join(data_dir, f"{prefix}_{timestamp}")

    # --- CSV (all points) ---
    fieldnames = ['place_id', 'name', 'address', 'lat', 'lng', 'types', 'keyword',
                  'search_lat', 'search_lon', 'rating', 'user_ratings_total', 'source', 'language', 'category']
    csv_file = f"{base}.csv"
    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(all_results)
    print(f"✅ CSV saved: {csv_file}")

    # --- JSON (all points) ---
    json_file = f"{base}.json"
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    print(f"✅ JSON saved: {json_file}")

    # --- GeoPackage (two layers) ---
    try:
        import geopandas as gpd
        import pandas as pd
        from shapely.geometry import Point

        keep_cols = ['place_id', 'name', 'address', 'types', 'keyword', 'search_lat', 'search_lon',
                     'rating', 'user_ratings_total', 'source', 'language', 'category']

        # ---- Filling points ----
        if filling_results:
            df_f = pd.DataFrame(filling_results)
            geometry_f = [Point(x, y) for x, y in zip(df_f['lng'], df_f['lat'])]
            available_f = [c for c in keep_cols if c in df_f.columns]
            gdf_f = gpd.GeoDataFrame(df_f[available_f], geometry=geometry_f, crs="EPSG:4326")
            gdf_f['lat'] = df_f['lat']
            gdf_f['lon'] = df_f['lng']
            gdf_f_3857 = gdf_f.to_crs("EPSG:3857")
            gpkg_filling = f"{base}_filling_3857.gpkg"
            gdf_f_3857.to_file(gpkg_filling, driver='GPKG', layer='filling_points')
            print(f"✅ GeoPackage (filling) saved: {gpkg_filling}")
        else:
            print("⚠️ No filling points, skipping filling GeoPackage.")

        # ---- Reseller points ----
        if reseller_results:
            df_r = pd.DataFrame(reseller_results)
            geometry_r = [Point(x, y) for x, y in zip(df_r['lng'], df_r['lat'])]
            available_r = [c for c in keep_cols if c in df_r.columns]
            gdf_r = gpd.GeoDataFrame(df_r[available_r], geometry=geometry_r, crs="EPSG:4326")
            gdf_r['lat'] = df_r['lat']
            gdf_r['lon'] = df_r['lng']
            gdf_r_3857 = gdf_r.to_crs("EPSG:3857")
            gpkg_reseller = f"{base}_reseller_3857.gpkg"
            gdf_r_3857.to_file(gpkg_reseller, driver='GPKG', layer='reseller_points')
            print(f"✅ GeoPackage (reseller) saved: {gpkg_reseller}")
        else:
            print("⚠️ No reseller points, skipping reseller GeoPackage.")

    except ImportError:
        print("⚠️ GeoPandas not installed, skipping GeoPackage.")
        print("   Install with: pip install geopandas pandas shapely")

# ==================== GOOGLE STATISTICS FUNCTIONS ====================

def print_google_stats(google_results):
    if not google_results:
        print("No Google results to analyze.")
        return
    print("\n=== GOOGLE STATISTICS ===")
    lang_counter = Counter(r['language'] for r in google_results)
    print("Results by language:")
    for lang, cnt in lang_counter.most_common():
        print(f"  {lang}: {cnt}")
    kw_counter = Counter()
    for r in google_results:
        kw_full = r['keyword']
        kw_orig = kw_full.rsplit('_', 1)[0]
        kw_counter[kw_orig] += 1
    print("Results by keyword:")
    for kw, cnt in kw_counter.most_common():
        print(f"  {kw}: {cnt}")

def haversine(lat1, lon1, lat2, lon2):
    """Distance in metres between two coordinates."""
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def compute_overlap(osm_points, google_points, max_dist=300):
    """
    Find OSM and Google points within max_dist metres.
    Returns: set of osm_ids, set of google_ids that have at least one match,
    and list of pairs (osm_id, google_id, dist).
    """
    matched_osm = set()
    matched_google = set()
    pairs = []
    for g in google_points:
        for o in osm_points:
            d = haversine(o['lat'], o['lng'], g['lat'], g['lng'])
            if d <= max_dist:
                matched_osm.add(o['place_id'])
                matched_google.add(g['place_id'])
                pairs.append((o['place_id'], g['place_id'], d))
    return matched_osm, matched_google, pairs


def generate_grid(bbox, step):
    points = []
    lat = bbox['min_lat']
    while lat <= bbox['max_lat']:
        lon = bbox['min_lon']
        while lon <= bbox['max_lon']:
            points.append((round(lat, 4), round(lon, 4)))
            lon += step
        lat += step
    return points
def print_rating_stats(all_results, label="ALL"):
    """Print detailed rating and review statistics for a given set of results."""
    if not all_results:
        print(f"\n--- Rating stats ({label}): no points ---")
        return
    # Convert rating and review counts to numeric
    for r in all_results:
        try:
            r['rating_num'] = float(r['rating']) if r['rating'] else None
        except:
            r['rating_num'] = None
        try:
            r['reviews_num'] = float(r['user_ratings_total']) if r['user_ratings_total'] else 0.0
        except:
            r['reviews_num'] = 0.0

    with_rating = [r for r in all_results if r['rating_num'] is not None]
    print(f"\n=== RATING & REVIEW STATS ({label}) ===")
    print(f"Total points: {len(all_results)}")
    print(f"Points with rating: {len(with_rating)}")
    if with_rating:
        ratings = [r['rating_num'] for r in with_rating]
        avg_rating = sum(ratings) / len(ratings)
        median_rating = sorted(ratings)[len(ratings)//2]
        print(f"Average rating: {avg_rating:.2f}")
        print(f"Median rating: {median_rating}")
        bins = [(0,1), (1,2), (2,3), (3,4), (4,5)]
        print("Rating distribution (points with rating):")
        for low, high in bins:
            count = sum(1 for r in with_rating if low <= r['rating_num'] < high or (high==5 and r['rating_num']==5))
            print(f"  {low} ≤ rating < {high}: {count}")
        low_rating = [r for r in with_rating if r['rating_num'] < 2]
        print(f"Points with rating < 2: {len(low_rating)}")

    reviews_all = [r['reviews_num'] for r in all_results]
    avg_rev = sum(reviews_all) / len(reviews_all) if reviews_all else 0
    median_rev = sorted(reviews_all)[len(reviews_all)//2] if reviews_all else 0
    print(f"Review count (all points): average {avg_rev:.1f}, median {median_rev}")
    review_bins = [(0,2), (2,5), (5,10), (10,50), (50,100), (100,999999)]
    print("Review distribution:")
    for low, high in review_bins:
        count = sum(1 for r in all_results if low <= r['reviews_num'] < high)
        print(f"  {low} ≤ reviews < {high}: {count}")
    few_reviews = [r for r in all_results if r['reviews_num'] < 2]
    print(f"Points with < 2 reviews: {len(few_reviews)}")
    combined = [r for r in all_results if r['rating_num'] is not None and r['rating_num'] < 2 and r['reviews_num'] < 2]
    print(f"Points with rating < 2 AND reviews < 2: {len(combined)}")
# ==================== MAIN ====================
def main():
    print(f"=== LPG DATA COLLECTION - {CITY_NAME.replace('_', ' ').upper()} ===")
    grid = generate_grid(DAR_BBOX, STEP)
    print(f"Grid points: {len(grid)}")

    all_places = {}
    total_osm_calls = 0
    total_google_calls = 0
    error_count = 0

    for idx, (lat, lon) in enumerate(grid):
        if idx % 3 == 0 or idx == 0:
            print(f"Point {idx+1}/{len(grid)}: ({lat}, {lon})")

        # ---- OSM always active ----
        try:
            osm_places = osm_search(lat, lon, RADIUS)
            total_osm_calls += 1
            for p in osm_places:
                if p['place_id'] not in all_places:
                    all_places[p['place_id']] = p
            time.sleep(1)  # respect Overpass API
        except Exception as e:
            error_count += 1
            print(f"  ⚠️ OSM error: {e}")

        # ---- Google optional ----
        if USE_GOOGLE:
            # priority to filling
            for lang, kwlist in FILLING_KEYWORDS.items():
                for kw in kwlist:
                    try:
                        places = google_nearby_search(lat, lon, kw, lang, category='filling')
                        total_google_calls += 1 + (len(places)//60)
                        for p in places:
                            pid = p['place_id']
                            if pid not in all_places:
                                all_places[pid] = p
                            else:
                                if all_places[pid].get('category') != 'filling':
                                    all_places[pid]['category'] = 'filling'
                    except Exception as e:
                        error_count += 1
                        print(f"  ⚠️ error Google filling {kw}_{lang}: {e}")
                    time.sleep(0.2)

            # then reseller
            for lang, kwlist in RESELLER_KEYWORDS.items():
                for kw in kwlist:
                    try:
                        places = google_nearby_search(lat, lon, kw, lang, category='reseller')
                        total_google_calls += 1 + (len(places)//60)
                        for p in places:
                            pid = p['place_id']
                            if pid not in all_places:
                                all_places[pid] = p
                            else:
                                # mantain 'filling' if present
                                if all_places[pid].get('category') != 'filling':
                                    pass  #leave as is (reseller)
                    except Exception as e:
                        error_count += 1
                        print(f"  ⚠️ error Google reseller {kw}_{lang}: {e}")
                    time.sleep(0.2)

        # Checkpoint
        if (idx + 1) % 5 == 0:
            interim_path = os.path.join(DATA_DIR, f"{CITY_NAME}_interim.json")
            with open(interim_path, 'w') as f:
                json.dump(list(all_places.values()), f, indent=2)
            print(f"  [Checkpoint] {len(all_places)} unique places")

    print("\n=== COLLECTION COMPLETED ===")
    print(f"OSM calls: {total_osm_calls}")
    if USE_GOOGLE:
        print(f"Estimated Google calls: {total_google_calls}")
    print(f"Total unique places: {len(all_places)}")
    print(f"Errors: {error_count}")

    results = list(all_places.values())
    osm_results = [r for r in results if r['source'] == 'osm']
    google_results = [r for r in results if r['source'] == 'google']
    print(f"\nDi cui OSM: {len(osm_results)}")
    print(f"Di cui Google: {len(google_results)}")

    # --- Categorization ---
    filling_results = [r for r in results if r.get('category') == 'filling']
    reseller_results = [r for r in results if r.get('category') != 'filling']  # includes OSM
    print(f"\n=== CATEGORIES ===")
    print(f"Filling points: {len(filling_results)}")
    print(f"Reseller points: {len(reseller_results)}")

    filling_osm = [r for r in filling_results if r['source'] == 'osm']
    filling_google = [r for r in filling_results if r['source'] == 'google']
    print(f"  - Filling from OSM: {len(filling_osm)}")
    print(f"  - Filling from Google: {len(filling_google)}")

    # --- Google language & keyword stats ---
    if USE_GOOGLE and google_results:
        print("\n=== GOOGLE STATS (all Google points) ===")
        print_google_stats(google_results)

    # --- Overlap OSM/Google (using all google points) ---
    if USE_GOOGLE and osm_results and google_results:
        print("\n=== OVERLAP OSM/GOOGLE (max 300 m) ===")
        matched_osm_ids, matched_google_ids, pairs = compute_overlap(osm_results, google_results, max_dist=300)
        print(f"OSM points with a Google match within 300 m: {len(matched_osm_ids)}")
        print(f"Google points with an OSM match within 300 m: {len(matched_google_ids)}")
        print(f"Number of matched pairs: {len(pairs)}")
        print(f"OSM points without match: {len(osm_results) - len(matched_osm_ids)}")
        print(f"Google points without match: {len(google_results) - len(matched_google_ids)}")

    # --- Rating and review statistics ---
    # All points
    print_rating_stats(results, label="ALL POINTS")
    # Filling only
    print_rating_stats(filling_results, label="FILLING POINTS")
    # Reseller only
    print_rating_stats(reseller_results, label="RESELLER POINTS")

    # --- Save summary statistics ---
    summary_stats = {
        'city': CITY_NAME,
        'total_unique': len(all_places),
        'osm_count': len(osm_results),
        'google_count': len(google_results),
        'filling_count': len(filling_results),
        'reseller_count': len(reseller_results),
        'osm_matched': len(matched_osm_ids) if USE_GOOGLE and osm_results and google_results else 0,
        'google_matched': len(matched_google_ids) if USE_GOOGLE and osm_results and google_results else 0,
        'pairs': len(pairs) if USE_GOOGLE and osm_results and google_results else 0
    }
    stats_file = os.path.join(DATA_DIR, f"{CITY_NAME}_statistics.json")
    with open(stats_file, 'w') as f:
        json.dump(summary_stats, f, indent=2)
    print(f"\nSummary statistics saved to {stats_file}")

    # --- Save results (CSV, JSON, and two GeoPackages) ---
    save_results(results, filling_results, reseller_results, DATA_DIR, f"{CITY_NAME}_lpg")

# ==================== LISTA CITTÀ DA PROCESSARE ====================
cities_to_process = [
    
    #{"name": "dar_es_salaam", "bbox": {'min_lat': -7.2, 'max_lat': -6.4, 'min_lon': 38.9, 'max_lon': 39.6}},
    {"name": "kampala", "bbox": {'min_lat': 0.25, 'max_lat': 0.45, 'min_lon': 32.50, 'max_lon': 32.70}},
    {"name": "lusaka", "bbox": {'min_lat': -15.50, 'max_lat': -15.30, 'min_lon': 28.20, 'max_lon': 28.40}},
    {"name": "harare", "bbox": {'min_lat': -17.90, 'max_lat': -17.70, 'min_lon': 30.95, 'max_lon': 31.15}},
    {"name": "yaounde", "bbox": {'min_lat': 3.80, 'max_lat': 4.00, 'min_lon': 11.45, 'max_lon': 11.60}},
    {"name": "abidjan", "bbox": {'min_lat': 5.25, 'max_lat': 5.45, 'min_lon': -4.10, 'max_lon': -3.90}},
    {"name": "juba", "bbox": {'min_lat': 4.75, 'max_lat': 4.95, 'min_lon': 31.50, 'max_lon': 31.70}},
    {"name": "beira", "bbox": {'min_lat': -19.90, 'max_lat': -19.70, 'min_lon': 34.75, 'max_lon': 34.95}},
    {"name": "blantyre", "bbox": {'min_lat': -15.85, 'max_lat': -15.65, 'min_lon': 34.90, 'max_lon': 35.10}},
    {"name": "bobo_dioulasso", "bbox": {'min_lat': 11.10, 'max_lat': 11.30, 'min_lon': -4.35, 'max_lon': -4.15}},
    {"name": "zinder", "bbox": {'min_lat': 13.70, 'max_lat': 13.90, 'min_lon': 8.90, 'max_lon': 9.10}},
]

# ==================== ESECUZIONE IN LOOP ====================
for city_config in cities_to_process:
    # Aggiorna le variabili globali per la città corrente
    CITY_NAME = city_config["name"]
    DAR_BBOX = city_config["bbox"]
    DATA_DIR = os.path.join("other_city", CITY_NAME)
    os.makedirs(DATA_DIR, exist_ok=True)

    print("\n" + "="*60)
    print(f"INIZIO LAVORAZIONE: {CITY_NAME}")
    print("="*60)

    # Controllo chiave Google (già definita sopra)
    if USE_GOOGLE and API_KEY == "LA_TUA_API_KEY_GOOGLE":
        print("❌ USE_GOOGLE=True ma API_KEY non impostata. Google disabilitato per questa run.")
        USE_GOOGLE_LOCAL = False
    else:
        USE_GOOGLE_LOCAL = USE_GOOGLE

    
    main()

    print(f"FINE LAVORAZIONE: {CITY_NAME}\n")
    # Pausa tra una città e l'altra per rispettare i limiti delle API
    time.sleep(5)  # 5 secondi di pausa; puoi regolare, con 20 funzionava bene


INIZIO LAVORAZIONE: kampala
=== LPG DATA COLLECTION - KAMPALA ===
Grid points: 4
Point 1/4: (0.25, 32.5)
Point 4/4: (0.45, 32.7)

=== COLLECTION COMPLETED ===
OSM calls: 4
Estimated Google calls: 224
Total unique places: 225
Errors: 0

Di cui OSM: 6
Di cui Google: 219

=== CATEGORIES ===
Filling points: 74
Reseller points: 151
  - Filling from OSM: 0
  - Filling from Google: 74

=== GOOGLE STATS (all Google points) ===

=== GOOGLE STATISTICS ===
Results by language:
  en: 218
  pt: 1
Results by keyword:
  cylinder filling station: 64
  gas station with LPG: 43
  LPG refill station: 22
  gas cylinder exchange: 22
  LPG filling point: 21
  cooking gas: 20
  LPG: 17
  LPG bottling station: 7
  LPG dealer: 2
  estação de recarga de GLP: 1

=== OVERLAP OSM/GOOGLE (max 300 m) ===
OSM points with a Google match within 300 m: 3
Google points with an OSM match within 300 m: 3
Number of matched pairs: 3
OSM points without match: 3
Google points without match: 216

=== RATING & REVIEW STATS (ALL